# M5 — GP Ensemble Diagnostic (Manuscript, Supervisor Feedback Item A-1)

**Purpose.** Quantify how much discrimination is lost by insisting on a single auditable GP formula, versus a small
team of formulas averaged together. Addresses supervisor feedback item A-1.

**Plan.**
1. Load all 30 marathon-run GP models (`results/v2_bce/gp_runs/run_00..29_model.pkl`) and their validation summary
   (`results/v2_bce/tables/gp_30runs_summary.csv`). All 30 are used — none are degenerate (checked individually;
   val AUROC ranges 0.730-0.764, complexity ranges 18-24). The previously-noted degenerate run
   (complexity 6, AUROC 0.6616) belongs to a *different* experiment (`NB11_repro_runs.ipynb`'s 30 resplit runs),
   not this marathon set, and is not part of this analysis.
2. Rank the 30 runs by the validation criterion phi = val_auroc - 0.5*val_ece (already stored as `val_fitness`
   in `gp_30runs_summary.csv`).
3. For each run, load its saved model and predict on: (a) the i.i.d. test set, (b) the 13-hospital test partition.
   No retraining — these are the already-evolved expressions, scored forward on held-out patients.
4. For k in {1, 3, 5, 10, 30}, average the sigmoid-transformed predictions of the top-k runs (by phi) per patient,
   and compute AUROC / ECE / Brier on both test sets.
5. Sanity-check: k=1 (seed=14 alone) should reproduce the published canonical test-set numbers
   (AUROC 0.738, ECE 0.013, Brier 0.120).

In [1]:
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT = Path(r"C:\ML PROJECT\sepsis-gp")
sys.path.insert(0, str(PROJECT))
from src.metrics import compute_metrics

RUNS_DIR = PROJECT / "results" / "v2_bce" / "gp_runs"
TABLES_SRC = PROJECT / "results" / "v2_bce" / "tables"
OUT_DIR = PROJECT / "results" / "manuscript" / "tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(PROJECT / "data" / "processed" / "feature_config.json") as f:
    cfg = json.load(f)
GP_TERMINALS = cfg.get("GP_TERMINALS")
if not GP_TERMINALS:
    term_df = pd.read_csv(PROJECT / "data" / "processed" / "gp_terminal_set.csv")
    GP_TERMINALS = term_df["feature"].tolist()
print(f"{len(GP_TERMINALS)} GP terminals loaded")

26 GP terminals loaded


## Step 1 — Load test-set patients (i.i.d. test set and 13-hospital test partition)

In [2]:
feat = pd.read_parquet(PROJECT / "data" / "processed" / "features_curated.parquet")
split_iid = pd.read_csv(PROJECT / "data" / "processed" / "split_random.csv")
split_hosp = pd.read_csv(PROJECT / "data" / "processed" / "split_hospital_out.csv")

feat_iid = feat.merge(split_iid[["patientunitstayid", "split"]], on="patientunitstayid")
test_iid = feat_iid[feat_iid["split"] == "test"].reset_index(drop=True)

feat_hosp = feat.merge(split_hosp[["patientunitstayid", "split"]], on="patientunitstayid")
test_hosp = feat_hosp[feat_hosp["split"] == "test"].reset_index(drop=True)

X_test_iid = test_iid[GP_TERMINALS]
y_test_iid = test_iid["hospital_mortality"].to_numpy(dtype=np.float64)

X_test_hosp = test_hosp[GP_TERMINALS]
y_test_hosp = test_hosp["hospital_mortality"].to_numpy(dtype=np.float64)

print(f"i.i.d. test set: {len(X_test_iid):,} patients")
print(f"13-hospital test partition: {len(X_test_hosp):,} patients, {test_hosp['hospitalid'].nunique()} hospitals")

i.i.d. test set: 2,233 patients
13-hospital test partition: 2,062 patients, 13 hospitals


## Step 2 — Rank the 30 marathon runs by validation criterion phi (val_fitness)

In [3]:
summary = pd.read_csv(TABLES_SRC / "gp_30runs_summary.csv")
summary_sorted = summary.sort_values("val_fitness", ascending=False).reset_index(drop=True)
summary_sorted[["seed", "complexity", "val_auroc", "val_ece", "val_fitness"]]

,seed,complexity,val_auroc,val_ece,val_fitness
0,14,24,0.7640,0.0211,0.7535
1,25,23,0.7592,0.0120,0.7532
2,12,23,0.7559,0.0085,0.7517
3,6,21,0.7553,0.0117,0.7495
4,21,24,0.7556,0.0150,0.7482
5,0,24,0.7558,0.0154,0.7481
6,3,23,0.7632,0.0303,0.7481
7,8,23,0.7524,0.0110,0.7469
8,13,23,0.7573,0.0209,0.7469
9,11,24,0.7563,0.0193,0.7467


## Step 3 — Load each run's fixed model, predict on both test sets (no retraining)

In [4]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

prob_iid = {}
prob_hosp = {}

for _, rec in summary_sorted.iterrows():
    seed = int(rec["seed"])
    complexity = int(rec["complexity"])

    with open(RUNS_DIR / f"run_{seed:02d}_model.pkl", "rb") as f:
        model = pickle.load(f)

    match = model.equations_[model.equations_["complexity"] == complexity]
    if len(match) == 0:
        print(f"WARNING seed={seed}: no exact complexity={complexity} match")
        continue
    eq_idx = match.index[0]

    raw_iid = model.predict(X_test_iid, index=eq_idx)
    raw_iid = np.where(np.isfinite(raw_iid), raw_iid, 0.0)
    prob_iid[seed] = sigmoid(raw_iid)

    raw_hosp = model.predict(X_test_hosp, index=eq_idx)
    raw_hosp = np.where(np.isfinite(raw_hosp), raw_hosp, 0.0)
    prob_hosp[seed] = sigmoid(raw_hosp)

    print(f"seed={seed:2d} complexity={complexity:2d} scored OK")

print(f"\nScored {len(prob_iid)} / 30 runs successfully")

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


seed=14 complexity=24 scored OK
seed=25 complexity=23 scored OK
seed=12 complexity=23 scored OK
seed= 6 complexity=21 scored OK
seed=21 complexity=24 scored OK
seed= 0 complexity=24 scored OK
seed= 3 complexity=23 scored OK
seed= 8 complexity=23 scored OK
seed=13 complexity=23 scored OK


seed=11 complexity=24 scored OK
seed=28 complexity=21 scored OK
seed= 7 complexity=23 scored OK


seed=22 complexity=20 scored OK
seed=19 complexity=21 scored OK
seed=20 complexity=24 scored OK
seed=23 complexity=19 scored OK
seed=10 complexity=23 scored OK
seed=26 complexity=23 scored OK
seed=29 complexity=22 scored OK
seed= 4 complexity=21 scored OK


seed=24 complexity=21 scored OK


seed= 1 complexity=23 scored OK
seed=18 complexity=24 scored OK
seed=17 complexity=24 scored OK
seed=27 complexity=24 scored OK


seed= 5 complexity=23 scored OK
seed= 2 complexity=18 scored OK
seed=16 complexity=23 scored OK
seed= 9 complexity=24 scored OK
seed=15 complexity=23 scored OK

Scored 30 / 30 runs successfully


## Step 4 — Sanity check: does k=1 (seed=14 alone) reproduce the published canonical numbers?

In [5]:
canonical_seed = 14
m_iid = compute_metrics(y_test_iid, prob_iid[canonical_seed], label="GP canonical, i.i.d. test")
m_hosp = compute_metrics(y_test_hosp, prob_hosp[canonical_seed], label="GP canonical, 13-hospital test")
print("=== SANITY CHECK: seed=14 alone vs published manuscript numbers ===")
print(f"i.i.d. test  -> computed AUROC={m_iid['auroc']}, ECE={m_iid['ece_10bin']}, Brier={m_iid['brier']}")
print(f"               published  AUROC=0.738,        ECE=0.013,        Brier=0.120")
print(f"13-hosp test -> computed AUROC={m_hosp['auroc']}, ECE={m_hosp['ece_10bin']}, Brier={m_hosp['brier']}")
print(f"               published(memory) AUROC=0.745, ECE=0.017")

=== SANITY CHECK: seed=14 alone vs published manuscript numbers ===
i.i.d. test  -> computed AUROC=0.7381, ECE=0.0126, Brier=0.1202
               published  AUROC=0.738,        ECE=0.013,        Brier=0.120
13-hosp test -> computed AUROC=0.7453, ECE=0.0168, Brier=0.1292
               published(memory) AUROC=0.745, ECE=0.017


## Step 5 — Build ensembles for k = 1, 3, 5, 10, 30 and evaluate

In [6]:
ordered_seeds = [s for s in summary_sorted["seed"].astype(int).tolist() if s in prob_iid]

results = []
for k in [1, 3, 5, 10, len(ordered_seeds)]:
    top_k = ordered_seeds[:k]
    ens_iid = np.mean([prob_iid[s] for s in top_k], axis=0)
    ens_hosp = np.mean([prob_hosp[s] for s in top_k], axis=0)
    m1 = compute_metrics(y_test_iid, ens_iid, label=f"GP ensemble k={k} (iid)")
    m2 = compute_metrics(y_test_hosp, ens_hosp, label=f"GP ensemble k={k} (hosp)")
    results.append({
        "k": k, "seeds": top_k,
        "iid_auroc": m1["auroc"], "iid_ece": m1["ece_10bin"], "iid_brier": m1["brier"],
        "hosp_auroc": m2["auroc"], "hosp_ece": m2["ece_10bin"], "hosp_brier": m2["brier"],
    })

res_df = pd.DataFrame(results)
res_df[["k", "iid_auroc", "iid_ece", "iid_brier", "hosp_auroc", "hosp_ece", "hosp_brier"]]

,k,iid_auroc,iid_ece,iid_brier,hosp_auroc,hosp_ece,hosp_brier
0,1,0.7381,0.0126,0.1202,0.7453,0.0168,0.1292
1,3,0.7425,0.0181,0.1201,0.7489,0.0170,0.1286
2,5,0.7456,0.0135,0.1194,0.7524,0.0197,0.1281
3,10,0.7489,0.0166,0.1188,0.7551,0.0187,0.1279
4,30,0.7497,0.0148,0.1187,0.7568,0.0160,0.1277


In [7]:
out_path = OUT_DIR / "M5_ensemble_diagnostic.csv"
res_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

Saved: C:\ML PROJECT\sepsis-gp\results\manuscript\tables\M5_ensemble_diagnostic.csv


## Findings

The sanity check confirmed pipeline validity: the k=1 result (seed=14 alone) reproduced the published
canonical test-set metrics almost exactly (i.i.d. test: AUROC 0.7381 vs. published 0.738, ECE 0.0126 vs.
published 0.013, Brier 0.1202 vs. published 0.120; 13-hospital partition: AUROC 0.7453 vs. published 0.745,
ECE 0.0168 vs. published 0.017).

Ensembling the top-k GP expressions by validation criterion phi produced a monotonic AUROC improvement
with increasing k on both evaluation sets: from 0.738 (k=1) to 0.750 (k=30) on the i.i.d. test set, and from
0.745 (k=1) to 0.757 (k=30) on the 13-hospital partition. ECE remained low across ensemble sizes and did not
degrade meaningfully (0.013-0.018 on the i.i.d. test set; 0.016-0.020 on the hospital partition), so the
discrimination gain was not bought at the cost of calibration.

The discrimination gap relative to random forest (AUROC 0.777 i.i.d., 0.785 on the same 13-hospital
partition) narrowed but did not close. On the i.i.d. test set, the gap fell from 3.89 percentage points
(single formula) to 2.73 percentage points (30-formula ensemble) — a 29.8% reduction. On the 13-hospital
partition, the gap fell from 3.97 to 2.82 percentage points — a 29.0% reduction. Approximately 70% of the
discrimination gap therefore persisted regardless of ensemble size.

This is a partial-recovery outcome, between the two extremes anticipated: ensembling recovers a modest but
non-trivial share of the gap (roughly 30%, confirming the single-formula constraint contributes something
real), but the majority of the gap is attributable to a binding limitation elsewhere — most plausibly the
ten-variable terminal set or the complexity-24 cap, rather than the single-expression constraint alone. The
result is best framed as a quantified, partial cost of auditability: retaining one formula for interpretability
gives up roughly 30% of the AUROC that a 30-formula ensemble of the same search would deliver, while the
remaining ~70% reflects the feature/complexity budget rather than the single-formula choice itself.